# Module 08-2: Circuit Breaker 패턴 구현

## 학습 목표
- Circuit Breaker의 3가지 상태(CLOSED→OPEN→HALF_OPEN)를 이해하고 구현할 수 있다
- `CircuitState` enum과 `CircuitBreaker` 클래스를 작성할 수 있다
- 외부 API 호출에 Circuit Breaker를 적용하여 연속 실패 시 빠른 실패를 구현할 수 있다

**참조 문서**: `docs/part4-production/08-error-handling-resilience.md` 섹션 2.4

## 개념 요약

### Circuit Breaker 상태 전이

```
              연속 실패 >= failure_threshold
   CLOSED ─────────────────────────────────> OPEN
  (정상)                                    (차단)
     ^                                        |
     |                                        | recovery_timeout 경과
     |                                        v
     |        테스트 성공                   HALF_OPEN
     └─────────────────────────────────── (테스트 중)
                                              |
                                              | 테스트 실패
                                              v
                                            OPEN
```

| 상태 | 동작 |
|------|------|
| **CLOSED** | 모든 요청 통과, 실패 횟수 추적 |
| **OPEN** | 모든 요청 즉시 거부 (타임아웃 대기 없음) |
| **HALF_OPEN** | 1건만 통과시켜 복구 확인 |

### 왜 Circuit Breaker가 필요한가?

외부 시스템이 다운되면, 재시도마다 타임아웃(30초)을 기다립니다.  
5번 재시도 × 30초 = **150초** 낭비! Circuit Breaker는 이를 **즉시 거부**로 방지합니다.

In [ ]:
# 환경 설정
import sys
sys.path.insert(0, '..')

import time
import threading
from enum import Enum
from dataclasses import dataclass, field

from common.fake_llm import FakeLLM

print("환경 설정 완료")

## 실습 1: CircuitState Enum 정의

Circuit Breaker의 3가지 상태를 나타내는 `CircuitState` Enum을 정의합니다.

In [ ]:
# TODO: 여기에 코드를 작성하세요
#
# 힌트 1 (방향): Enum 클래스를 상속하여 CLOSED, OPEN, HALF_OPEN 3개의 상태를 정의하세요.
#               각 상태의 값은 문자열로 설정하면 디버깅 시 가독성이 좋습니다.
#
# 힌트 2 (키워드): class CircuitState(Enum), CLOSED = "closed", OPEN = "open", HALF_OPEN = "half_open"
#
# 힌트 3 (거의 정답):
#   class CircuitState(Enum):
#       CLOSED = "closed"
#       OPEN = "open"
#       HALF_OPEN = "half_open"

# TODO: CircuitState Enum 정의
CircuitState = None  # 이 줄을 지우고 Enum 클래스를 정의하세요


class CircuitOpenError(Exception):
    """Circuit Breaker가 OPEN 상태일 때 발생하는 예외."""
    def __init__(self, name: str, remaining_seconds: float):
        self.name = name
        self.remaining_seconds = remaining_seconds
        super().__init__(
            f"Circuit '{name}'이 OPEN 상태입니다. "
            f"{remaining_seconds:.1f}초 후 재시도하세요."
        )


print("CircuitState 및 CircuitOpenError 정의 완료")

In [ ]:
# 검증 셀
assert CircuitState is not None, "CircuitState Enum을 정의하세요"
assert hasattr(CircuitState, 'CLOSED'), "CircuitState.CLOSED가 없습니다"
assert hasattr(CircuitState, 'OPEN'), "CircuitState.OPEN이 없습니다"
assert hasattr(CircuitState, 'HALF_OPEN'), "CircuitState.HALF_OPEN이 없습니다"
assert CircuitState.CLOSED != CircuitState.OPEN
assert CircuitState.OPEN != CircuitState.HALF_OPEN
print(f"CircuitState 정의 통과: {[s.value for s in CircuitState]}")
print("실습 1 완료!")

## 실습 2: CircuitBreaker 클래스 구현

`CircuitBreaker` 클래스를 구현합니다.

핵심 메서드:
- `state` (property): 현재 상태 반환. OPEN → HALF_OPEN 자동 전환 포함
- `_record_success()`: 성공 시 failure_count 리셋, HALF_OPEN → CLOSED 전환
- `_record_failure()`: 실패 시 failure_count 증가, 임계값 초과 시 OPEN 전환
- `call(fn, *args, **kwargs)`: Circuit Breaker를 통해 함수 호출

In [ ]:
# TODO: 여기에 코드를 작성하세요
#
# 힌트 1 (방향): @dataclass로 CircuitBreaker를 정의합니다.
#               state property에서는 OPEN 상태일 때 recovery_timeout이 경과했는지 확인하여
#               자동으로 HALF_OPEN으로 전환합니다.
#               call()에서는 현재 state를 확인하고, OPEN이면 CircuitOpenError를 raise합니다.
#
# 힌트 2 (키워드): @dataclass, field(default=..., init=False), threading.Lock,
#               time.monotonic(), _failure_count >= failure_threshold -> OPEN
#
# 힌트 3 (거의 정답):
#   @property
#   def state(self) -> CircuitState:
#       with self._lock:
#           if self._state == CircuitState.OPEN:
#               elapsed = time.monotonic() - self._last_failure_time
#               if elapsed >= self.recovery_timeout:
#                   self._state = CircuitState.HALF_OPEN
#           return self._state
#
#   def call(self, fn, *args, **kwargs):
#       if self.state == CircuitState.OPEN:
#           remaining = self.recovery_timeout - (time.monotonic() - self._last_failure_time)
#           raise CircuitOpenError(self.name, max(0, remaining))
#       try:
#           result = fn(*args, **kwargs)
#           self._record_success()
#           return result
#       except self.expected_exceptions:
#           self._record_failure()
#           raise

@dataclass
class CircuitBreaker:
    """Circuit Breaker 구현.

    Args:
        name: 서킷 식별자
        failure_threshold: OPEN으로 전환되는 연속 실패 횟수
        recovery_timeout: OPEN에서 HALF_OPEN으로 전환되는 대기 시간(초)
        expected_exceptions: 실패로 카운트할 예외 타입
    """
    name: str
    failure_threshold: int = 5
    recovery_timeout: float = 60.0
    expected_exceptions: tuple = (Exception,)

    # 내부 상태
    _state: CircuitState = field(default=CircuitState.CLOSED, init=False)
    _failure_count: int = field(default=0, init=False)
    _last_failure_time: float = field(default=0.0, init=False)
    _lock: threading.Lock = field(default_factory=threading.Lock, init=False)

    @property
    def state(self) -> CircuitState:
        """현재 상태를 반환합니다. OPEN -> HALF_OPEN 자동 전환 포함."""
        # TODO: _lock으로 스레드 안전성 보장
        # TODO: OPEN 상태이고 recovery_timeout이 경과했으면 HALF_OPEN으로 전환
        pass  # 이 줄을 지우고 구현하세요

    def _record_success(self) -> None:
        """성공을 기록합니다."""
        # TODO: failure_count 리셋, HALF_OPEN -> CLOSED 전환
        pass

    def _record_failure(self) -> None:
        """실패를 기록합니다."""
        # TODO: failure_count 증가, last_failure_time 기록
        # TODO: failure_count >= failure_threshold이면 OPEN 전환
        pass

    def call(self, fn, *args, **kwargs):
        """Circuit Breaker를 통해 함수를 호출합니다."""
        # TODO: OPEN이면 CircuitOpenError raise
        # TODO: 성공 시 _record_success(), 실패 시 _record_failure()
        pass


print("CircuitBreaker 클래스 정의 완료")

In [ ]:
# 검증 셀: CLOSED -> OPEN 전환
cb = CircuitBreaker(
    name="test-api",
    failure_threshold=3,
    recovery_timeout=60.0,
    expected_exceptions=(RuntimeError,),
)

assert cb.state == CircuitState.CLOSED, "초기 상태는 CLOSED여야 합니다"

# 3번 실패 → OPEN으로 전환
for i in range(3):
    try:
        cb.call(lambda: (_ for _ in ()).throw(RuntimeError("실패")))
    except RuntimeError:
        pass

assert cb.state == CircuitState.OPEN, f"3번 실패 후 OPEN이어야 합니다. 현재: {cb.state}"
print(f"CLOSED -> OPEN 전환 통과: {cb._failure_count}번 실패 후 OPEN")

# OPEN 상태에서 요청 거부 확인
try:
    cb.call(lambda: "이 코드는 실행되면 안 됩니다")
    assert False, "CircuitOpenError가 발생해야 합니다"
except CircuitOpenError as e:
    print(f"OPEN 상태에서 즉시 거부 확인: {e}")

print("실습 2 검증 1 완료!")

In [ ]:
# 검증 셀: OPEN -> HALF_OPEN -> CLOSED 전환
cb2 = CircuitBreaker(
    name="recovery-test",
    failure_threshold=2,
    recovery_timeout=0.1,  # 테스트용으로 짧게 설정
    expected_exceptions=(RuntimeError,),
)

# 2번 실패 → OPEN
for _ in range(2):
    try:
        cb2.call(lambda: (_ for _ in ()).throw(RuntimeError("실패")))
    except RuntimeError:
        pass

assert cb2.state == CircuitState.OPEN

# recovery_timeout 경과 후 HALF_OPEN으로 자동 전환 확인
time.sleep(0.15)
assert cb2.state == CircuitState.HALF_OPEN, (
    f"recovery_timeout 경과 후 HALF_OPEN이어야 합니다. 현재: {cb2.state}"
)
print(f"OPEN -> HALF_OPEN 자동 전환 통과")

# HALF_OPEN에서 성공 → CLOSED 복구
result = cb2.call(lambda: "복구 성공")
assert result == "복구 성공"
assert cb2.state == CircuitState.CLOSED, f"성공 후 CLOSED여야 합니다. 현재: {cb2.state}"
print(f"HALF_OPEN -> CLOSED 복구 통과")
print("실습 2 완료!")

## 실습 3: FakeLLM과 CircuitBreaker 연동

FakeLLM 호출에 CircuitBreaker를 적용합니다.  
연속 실패 후 서킷이 열리고, 일정 시간 후 자동으로 복구되는 흐름을 확인합니다.

In [ ]:
# TODO: 여기에 코드를 작성하세요
#
# 힌트 1 (방향): CircuitBreaker를 생성하고, 일부러 실패를 유도하여 OPEN 상태로 만드세요.
#               OPEN 상태에서는 CircuitOpenError가 발생합니다.
#
# 힌트 2 (키워드): CircuitBreaker(name="llm-api", failure_threshold=3, recovery_timeout=0.1),
#               cb.call(lambda: ...), try/except CircuitOpenError
#
# 힌트 3 (거의 정답):
#   llm_cb = CircuitBreaker("llm-api", failure_threshold=3, recovery_timeout=0.1)
#   def call_llm_safely(prompt):
#       try:
#           return llm_cb.call(lambda: llm.invoke(prompt).content)
#       except CircuitOpenError as e:
#           return f"서킷 열림: {e.remaining_seconds:.1f}초 후 재시도"

llm = FakeLLM(responses={"분석": "분석 완료", "hello": "안녕하세요"})

# TODO: CircuitBreaker 생성
llm_cb = None  # 이 줄을 지우고 CircuitBreaker를 생성하세요

# TODO: 3번 ConnectionError 발생 → 서킷 OPEN
for i in range(3):
    try:
        # TODO: llm_cb.call()로 ConnectionError 발생
        pass
    except ConnectionError:
        print(f"실패 {i+1}회: 연결 오류")

print(f"현재 서킷 상태: {llm_cb.state if llm_cb else 'N/A'}")

In [ ]:
# 검증 셀
assert llm_cb is not None, "llm_cb가 None입니다. CircuitBreaker를 생성하세요"
assert llm_cb.state == CircuitState.OPEN, (
    f"3번 실패 후 OPEN이어야 합니다. 현재: {llm_cb.state}"
)

# OPEN 상태에서 요청하면 CircuitOpenError
try:
    llm_cb.call(lambda: llm.invoke("hello").content)
    assert False, "CircuitOpenError가 발생해야 합니다"
except CircuitOpenError as e:
    print(f"서킷 열림 확인: {e}")

print(f"CircuitBreaker 연동 테스트 통과")
print("실습 3 완료! 모든 실습 완료")

## 정리

이번 실습에서 학습한 내용:

1. **CircuitState Enum**: CLOSED(정상), OPEN(차단), HALF_OPEN(테스트) 3개 상태
2. **상태 전이 규칙**:
   - CLOSED → OPEN: 연속 실패 >= `failure_threshold`
   - OPEN → HALF_OPEN: `recovery_timeout` 경과 (자동)
   - HALF_OPEN → CLOSED: 테스트 요청 성공
   - HALF_OPEN → OPEN: 테스트 요청 실패
3. **빠른 실패(Fail Fast)**: OPEN 상태에서 타임아웃 없이 즉시 `CircuitOpenError` 발생
4. **스레드 안전성**: `threading.Lock`으로 상태 전이 보호

**다음 실습**: `03_resilient_agent.ipynb` - Retry + Circuit Breaker를 통합한 에이전트